# 🟡 Solution: SwiGLU Activation

Reference solution for the SwiGLU activation function.

$$\text{SwiGLU}(x) = (xW_1) \cdot \text{Swish}(xW_{gate}) \cdot W_2$$

In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def swiglu(x, W1, W2, Wgate):
    """
    SwiGLU 门控激活函数。

    这是 LLaMA、PaLM 等现代大模型的 MLP 核心组件。
    核心思想：用一个线性投影的 Swish 激活版本，对另一个线性投影做门控（逐元素相乘）。

    参数:
        x: 输入张量, shape = (B, d_model)
            B 是 batch size, d_model 是模型隐藏维度。
        W1: 上投影权重, shape = (d_model, d_ff)
            将输入投影到更高维的隐藏空间 (d_ff 通常是 4 * d_model)。
        W2: 下投影权重, shape = (d_ff, d_model)
            将隐藏维度投影回 d_model。
        Wgate: 门控权重, shape = (d_model, d_ff)
            产生门控信号，与 W1 的输出逐元素相乘。

    返回: shape = (B, d_model) 的输出张量
    """
    # ---- Step 1: 线性投影 ----
    # h = x @ W1, shape: (B, d_model) @ (d_model, d_ff) = (B, d_ff)
    # 这是标准的上投影，把输入从 d_model 维映射到 d_ff 维
    h = x @ W1

    # ---- Step 2: 门控信号（Swish 激活） ----
    # gate_raw = x @ Wgate, shape: (B, d_ff)
    # 这是另一条独立的线性投影路径，产生门控信号
    gate = x @ Wgate

    # ---- Step 3: Swish 激活 ----
    # Swish(z) = z * sigmoid(z)
    # sigmoid(z) 输出 (0, 1)，所以 Swish 本质上是「z 被自身的一个软门控」
    # 与 ReLU 的区别：Swish 在负值区域不是硬截断为 0，而是有一个小的负值输出
    # 这使得梯度在负值区域也能流动，训练更平滑
    # torch.sigmoid(gate) 的 shape 与 gate 相同: (B, d_ff)
    swish_gate = gate * torch.sigmoid(gate)

    # ---- Step 4: 门控相乘 ----
    # h * swish_gate: 逐元素相乘, shape 仍是 (B, d_ff)
    # 这就是 "GLU" (Gated Linear Unit) 的核心：用一个投影的激活值去「门控」另一个投影
    # 直觉：sigmoid 输出接近 1 的维度被保留，接近 0 的维度被抑制
    hidden = h * swish_gate

    # ---- Step 5: 下投影 ----
    # hidden @ W2, shape: (B, d_ff) @ (d_ff, d_model) = (B, d_model)
    # 把隐藏维度压缩回原始模型维度
    return hidden @ W2

In [ ]:
# Verify
torch.manual_seed(0)
B, D, H = 2, 8, 16
x = torch.randn(B, D)
W1 = torch.randn(D, H)
W2 = torch.randn(H, D)
Wgate = torch.randn(D, H)

out = swiglu(x, W1, W2, Wgate)
print("Output shape:", out.shape)

gate = x @ Wgate
swish_gate = gate * torch.sigmoid(gate)
print("Gate (swish) sample values:", swish_gate[0, :4])

In [ ]:
from torch_judge import check
check("swiglu")

## 📝 核心思路总结

1. **SwiGLU = Swish + GLU**：两条并行的线性投影路径，一条产生值 (h)，一条经过 Swish 激活产生门控信号，逐元素相乘后再下投影。
2. **Swish vs ReLU**：`Swish(z) = z * sigmoid(z)` 在负值区域有小的非零输出，梯度更平滑，训练效果优于 ReLU。
3. **门控机制**：GLU 的核心思想是让网络自己学习「哪些维度该保留、哪些该抑制」，比固定的激活函数更灵活。
4. **参数量**：SwiGLU 有 3 个权重矩阵（W1, Wgate, W2），比标准 MLP 多一个，所以 d_ff 通常设为 `(2/3) * 4 * d_model` 来保持总参数量不变。